# Допуск: Darwin Gödel Machine: Open-Ended Evolution of Self-Improving Agents

**Статья:** J. Zhang, S. Hu, C. Lu, R. Lange, J. Clune —
<https://arxiv.org/abs/2505.22954v3>.
Задания сверены с версией v3 от 12.03.2026 (ICLR 2026).

5 заданий. Под заголовком каждого задания указан его тип:

- **вопрос по коду** — в коде нет отклонений: ответьте на вопросы через код и статью;
- **расхождение со статьёй** — найдите одно отклонение от описанного авторами метода;
- **неверный комментарий** — код корректен, но один комментарий утверждает неправду.

Для отклонения укажите место, объясните, что не так, сошлитесь на раздел, алгоритм,
рисунок или таблицу статьи и опишите исправление словами. Писать, запускать и отлаживать
код не нужно. Вопросы относятся только к заданиям без отклонений.

Общие импорты:

In [ ]:
import torch

`agent` — программа агента: Python-код инструментов и рабочего процесса вокруг
замороженной foundation model (FM). `agent.solve(repo, instruction)` возвращает patch
с предложенными изменениями репозитория; `agent.modify(agent)` создаёт новую версию
его собственной программы.
`evaluate` проверяет агента на бенчмарке и возвращает число от 0 до 1.
`agent.is_valid()` означает, что код компилируется и сохраняет способность редактировать
репозиторий. Функции, которые используются, но не показаны, считайте корректными.
Сниппеты сокращают механизмы статьи и не запускают настоящий эволюционный поиск.

## Задание 1. Что именно улучшает себя

**Тип задания:** вопрос по коду — в коде нет отклонений, ответьте на вопросы.

Одна самомодификация (§3, §4.1, прил. C.3). `diagnose` — отдельная диагностическая FM;
`clone_repo` создаёт изолированную копию программы агента. `build_agent` загружает
получившийся Python-код, используя переданную FM без изменения её весов.

In [ ]:
def self_modify(parent, evaluation_logs, diagnose):
    proposal = diagnose(evaluation_logs)
    own_repo = clone_repo(parent.codebase)
    instruction = make_self_improvement_instruction(proposal)
    patch = parent.solve(own_repo, instruction)
    own_repo.apply(patch)
    child = build_agent(own_repo, foundation_model=parent.foundation_model)
    return child

**Вопросы.**

1. Почему редактируется `parent.codebase`, а не репозиторий очередной задачи? Какие
   изменения инструментов и рабочего процесса реально обнаружены (§4.4, рис. 3,
   прил. F)? Приведите по одному примеру каждого вида.
2. Что в эксперименте остаётся заданным человеком, несмотря на термин «self-improving»?
   Укажите границы изменения FM и внешнего процесса поиска (§3, §6).
3. Какой компонент предлагает направление улучшения и какой реализует его? Чем проверка
   результата отличается от требования исходной Gödel Machine (§3, прил. C.3)?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 2. Пополнение архива

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Одна итерация DGM (§3, алгоритм 1 в прил. C.4). `select_parents` считайте правильным;
в архиве лежат пары `(agent, score)`. Найдите одно отклонение.

In [ ]:
def evolve_once(archive, benchmark, select_parents):
    parents = select_parents(archive)
    for parent, parent_score in parents:
        child = parent.modify(parent)
        score = evaluate(child, benchmark)
        if child.is_valid() and score > parent_score:
            archive.append((child, score))
    return archive

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 3. Выбор родителя

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Выбор одного родителя из уже созданного архива (прил. C.2, формулы 1–7; прил. A.3).
`functioning_children` содержит число потомков, сохранивших способность редактировать код,
для каждого агента. Агентов с идеальной оценкой не выбирают. Найдите одно отклонение от
основного варианта DGM.

In [ ]:
def select_parent(archive, functioning_children):
    eligible = [(i, agent, score) for i, (agent, score) in enumerate(archive)
                if score < 1.0]
    scores = torch.tensor([score for _, _, score in eligible])       # (K,)
    chosen = int(scores.argmax())
    return eligible[chosen][1]

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 4. Одна задача Polyglot

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Оценка одного агента на одной задаче Polyglot (§4.2). `hidden_tests` — эталонные тесты
бенчмарка; `feedback` содержит их диагностику. `fresh_repo` каждый раз возвращает чистую
копию задачи. Внутри `solve` агент может пользоваться собственными проверками.
Найдите одно отклонение от метрики, которую сообщают авторы DGM.

In [ ]:
def evaluate_task(agent, task, hidden_tests):
    patch = agent.solve(task.fresh_repo(), task.instruction)
    passed, feedback = hidden_tests(task, patch)
    if not passed:
        retry = task.instruction + "\nGround-truth test feedback:\n" + feedback
        patch = agent.solve(task.fresh_repo(), retry)
        passed, feedback = hidden_tests(task, patch)
    return float(passed)

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 5. Максимальная оценка за устранение галлюцинаций

**Тип задания:** неверный комментарий — код корректен, но один комментарий утверждает неправду.

В отдельном эксперименте (прил. H, рис. 8) авторы оптимизировали proxy-оценку устранения
галлюцинаций вызовов инструментов. У узлов 96 и 114 оценки 1.67 и 2.0. Ниже выбрана
программа с максимальной опубликованной оценкой; пересчитывать саму оценку не требуется.
Найдите один неверный комментарий.

In [ ]:
def choose_by_proxy():
    nodes = [96, 114]
    scores = torch.tensor([1.67, 2.0])
    winner = int(scores.argmax())
    # The perfect proxy score proves that node 114 eliminated hallucinated tool use.
    return nodes[winner], float(scores[winner])

**Ваш ответ:**

_Напишите ответ здесь._